In [4]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform

from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     RandomizedSearchCV, cross_val_score, validation_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("Titanic-Dataset.csv")

X = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()
y = df["Survived"]
title = df["Name"].str.extract(r",\s*([^.]+)\.", expand=False)
X["Title"] = title.where(title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)
print("Доля выживших: train %.3f, test %.3f" % (y_train.mean(), y_test.mean()))

num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Sex", "Embarked", "Title"]
prep_scaled = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder()), cat_cols))

(712, 8) (179, 8)
Доля выживших: train 0.383, test 0.385


## Гиперпараметры KNN

| Гиперпараметр | Смысл | Диапазон | Как влияет на модель |
|---|---|---|---|
| `n_neighbors` (`k`) | Количество ближайших соседей, участвующих в голосовании за класс объекта | `3 … 51`, целые нечётные значения | Малый `k` → модель чувствительна к шуму и выбросам, **переобучение**; большой `k` → локальные особенности сглаживаются, **недообучение**. Нечётное `k` уменьшает вероятность ничьей в бинарной классификации |
| `weights` | Способ учёта соседей при голосовании: `uniform` — все соседи имеют одинаковый вес; `distance` — ближайшие соседи влияют сильнее | `uniform`, `distance` | `uniform` создаёт равномерное голосование; `distance` сильнее учитывает ближайших соседей и может улучшить качество, если локальная близость информативна |
| `metric` / `p` | Метрика расстояния между объектами. При `metric='minkowski'`: `p=1` соответствует манхэттенскому расстоянию, `p=2` — евклидову | `p ∈ {1, 2}` | Манхэттенское расстояние суммирует абсолютные различия по признакам, евклидово сильнее учитывает большие различия по отдельным признакам. Выбор зависит от структуры данных |

Ожидание: `n_neighbors` и `weights` действуют **совместно** — при малом `k` голосование по расстоянию позволяет сильнее учитывать локальную структуру, а при большом `k` результат становится более сглаженным. Оптимальная область определяется перекрёстной проверкой (`cross-validation`) по выбранной метрике качества (**F1**).


## Кросс-валидация

Используется `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` на обучающей выборке (712 объектов).

**Почему 5 фолдов.** Это компромисс между смещением, дисперсией и временем. При 5 фолдах модель обучается на 80% train: этого достаточно, чтобы F1 на фолде не был совсем шумным. 10 фолдов дали бы более шумную оценку на фолде, и удвоили бы время; при 3-х фолдах оценка получается заметно пессимистичнее. Стратификация нужна из-за дисбаланса (38% положительных); `shuffle` — потому что порядок строк в датасете не гарантированно случаен. Один и тот же объект `cv` используется для всех моделей, поэтому они сравниваются на одних и тех же фолдах.

**Зачем нужна кросс-валидация.** Нужно оценить качество модели с данными гиперпараметрами на объектах, которых она не видела, не трогая test. CV делает это K раз на разных частях train и усредняет: каждый объект побывает в валидации, а по разбросу между фолдами видно, насколько оценка надёжна.

**Почему одного разбиения train/validation недостаточно.** Оценка по одному разбиению — это одна случайная величина с большой дисперсией. Перебирая сотни комбинаций, мы выберем ту, которой «повезло» на этом разбиении, а не ту, что лучше в целом.

**Что будет, если многократно выбирать гиперпараметры по test.** Test превращается в ещё одну валидационную выборку: максимум из многих зашумлённых оценок смещён вверх, а финальная оценка перестаёт быть честной — свободных данных, на которых её можно проверить, больше нет. Чем больше комбинаций перебрано, тем сильнее завышение. Ниже, в конце ноутбука, это показано на цифрах.

In [6]:
baseline = make_pipeline(prep_scaled, KNeighborsClassifier(n_neighbors=21, weights='uniform', p=2))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

base_cv = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="f1")

print(f"F1 по разбиениям: ", base_cv.round(3))
print("CV F1: %.4f ± %.4f" % (base_cv.mean(), base_cv.std()))


F1 по разбиениям:  [0.781 0.733 0.8   0.692 0.727]
CV F1: 0.7466 ± 0.0389


In [7]:
for i, (tr, va) in enumerate(cv.split(X_train, y_train), 1):
    print(f"fold {i}: train={len(tr)}, val={len(va)}, выживших в val={y_train.iloc[va].sum()}")

fold 1: train=569, val=143, выживших в val=55
fold 2: train=569, val=143, выживших в val=55
fold 3: train=570, val=142, выживших в val=55
fold 4: train=570, val=142, выживших в val=54
fold 5: train=570, val=142, выживших в val=54


### GRID SEARCH

In [39]:
knn_search = make_pipeline(prep_scaled, KNeighborsClassifier())

param_grid = {
    "kneighborsclassifier__n_neighbors": np.arange(3, 52, 2),
    "kneighborsclassifier__weights": ["uniform", "distance"],
    "kneighborsclassifier__p": [1, 2]
}

n_grid = np.prod([len(v) for v in param_grid.values()])
print("Кол-во комбинаций параметров в сетке: ", n_grid)
print("Кол-во обучений: ", n_grid*cv.get_n_splits())

grid = GridSearchCV(estimator=knn_search, param_grid=param_grid, cv=cv, scoring="f1", n_jobs=-1, return_train_score=True)
t0 = time.perf_counter()
grid.fit(X_train, y_train)
grid_time = time.perf_counter() - t0 

print("Время GridSearch: ", grid_time)
print("Лучшие параметры: ", grid.best_params_)
print("F1: ", grid.best_score_)

Кол-во комбинаций параметров в сетке:  100
Кол-во обучений:  500
Время GridSearch:  1.7231566999980714
Лучшие параметры:  {'kneighborsclassifier__n_neighbors': 3, 'kneighborsclassifier__p': 1, 'kneighborsclassifier__weights': 'uniform'}
F1:  0.7526797634941826


### Random Search

In [40]:
rand_search = RandomizedSearchCV(knn_search, param_grid, n_iter=n_grid, cv=cv, n_jobs=-1, random_state=RANDOM_STATE, scoring="f1")

t0 = time.perf_counter()
rand_search.fit(X_train, y_train)
rand_time = time.perf_counter() - t0 

print("Время RandomSearch: ", rand_time)
print("Лучшие параметры: ", rand_search.best_params_)
print("F1: ", rand_search.best_score_)

Время RandomSearch:  1.141823199999635
Лучшие параметры:  {'kneighborsclassifier__weights': 'uniform', 'kneighborsclassifier__p': 1, 'kneighborsclassifier__n_neighbors': 3}
F1:  0.7526797634941826


In [41]:
summary = pd.DataFrame({
    "Комбинаций": [n_grid, len(rand_search.cv_results_["params"])],
    "Обучений (×5 фолдов)": [n_grid * 5, n_grid * 5],
    "Лучший CV F1": [grid.best_score_, rand_search.best_score_],
    "Время, с": [grid_time, rand_time],
}, index=["Grid Search", "Random Search"]).round(4)
summary

,Комбинаций,Обучений (×5 фолдов),Лучший CV F1,"Время, с"
Grid Search,100,500,0.7527,1.7232
Random Search,100,500,0.7527,1.1418
